In [0]:
# =============================================================
# gold_writer.py
# Layer     : Gold
# Purpose   : Gold Delta writes only.
#             - Null handling (coalesce) before write
#             - Audit column injection
#             - Overwrite mode (Gold always full-refreshes)
#             - Row count confirmation
#
# STRICT RULE: No reads. No groupBy. No joins.
#              If you see agg() here → wrong file.
# =============================================================

from pyspark.sql.functions import (
    current_timestamp, coalesce, lit
)


# ── Null fill rules per Gold table ────────────────────────────────────────────
# Rule 2 — Explicit Null Handling:
# After full/left joins, pets or devices with no events on a day
# will have NULLs in metric columns. Dashboards show NULL as blank.
# We coalesce to 0 for all additive numeric metrics.

_NULL_FILL_MAP = {
    "gold_pet_owner_daily_summary": {
        "total_steps":           0,
        "total_distance_km":     0.0,
        "total_active_minutes":  0,
        "avg_heart_rate":        None,   # keep null — no device means no reading
        "avg_body_temperature":  None,
        "max_heart_rate":        None,
        "min_heart_rate":        None,
        "health_alert_count":    0,
        "total_food_grams":      0.0,
        "total_feed_sessions":   0,
        "total_water_ml":        0.0,
        "total_drink_sessions":  0,
    },
    "gold_manufacturer_device_summary": {
        "total_failures":        0,
        "avg_severity_score":    None,
        "total_downtime_minutes":0,
        "total_deployments":     0,
        "successful_deployments":0,
        "total_rollbacks":       0,
        "fw_success_rate_pct":   None,
        "avg_battery_pct":       None,
        "avg_signal_strength":   None,
        "signal_drop_count":     0,
    },
    "gold_sales_summary": {
        "total_units_sold":  0,
        "gross_revenue":     0.0,
        "net_revenue":       0.0,
        "total_discounts":   0.0,
        "discount_rate_pct": 0.0,
    }
}


def apply_null_defaults(df, gold_table_name: str):
    """
    Apply coalesce null-filling for numeric metrics.
    Keeps NULL for vitals (heart_rate, temperature) where
    NULL means the device category doesn't support that sensor
    — that is meaningful, not missing data.
    """
    fill_map = _NULL_FILL_MAP.get(gold_table_name, {})
    for col_name, default in fill_map.items():
        if col_name in df.columns and default is not None:
            df = df.withColumn(
                col_name,
                coalesce(df[col_name], lit(default))
            )
    return df


def add_gold_audit(df):
    """Add Gold-layer audit timestamp."""
    return df.withColumn("_gold_load_timestamp", current_timestamp())


def write_gold(df, gold_table_name: str) -> int:
    """
    Write to Gold Delta — always full overwrite.

    Gold tables are pre-aggregated views built from the entire
    Silver history. Overwrite is safe and correct here because:
        1. Gold is small (aggregated, not row-level)
        2. Incremental merge would be complex for multi-source joins
        3. Power BI reads from Gold — a clean refresh is predictable

    Args:
        df              : Final Gold DataFrame with audit column
        gold_table_name : e.g. "gold_pet_owner_daily_summary"

    Returns:
        Final Gold row count
    """
    gold_path = abfss("gold", gold_table_name)
    print(f"[gold_writer] Writing to: {gold_path}")

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .save(gold_path)
    )

    final_count = spark.read.format("delta").load(gold_path).count()
    print(f"[gold_writer] Rows written: {final_count:,}")
    return final_count

print("[gold_writer] Loaded.")